# Imports

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 890.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.0 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os


#Obtenção de dados

In [ ]:
#Config GEE

ee.Authenticate()
ID_DO_PROJETO = 'our-chassis-340523'
ee.Initialize(project=ID_DO_PROJETO)
NOME_PROJETO_GEE = 'AnaliseDaOrla'

In [ ]:
#Ponto Inicial
#ini_lon, ini_lat = -46.772260, -24.178458
ini_lon, ini_lat =-46.799506, -24.194184


In [ ]:
def convertToImageArray(img, aoi, normalize = True):
  img_array = geemap.ee_to_numpy(img, region=aoi, scale=10)
  img_array = np.nan_to_num(img_array)

  if not normalize:
    return img_array

  #Pega pontos de brilho extremo ou brilho muito baixo e os retira
  p2, p98 = np.percentile(img_array, (2, 98))
  if p98 - p2 > 0:
      img_scaled = np.clip((img_array - p2) / (p98 - p2), 0, 1)
  else:
      img_scaled = np.clip(img_array / 3000.0, 0, 1)

  return np.power(img_scaled, 0.65)


In [ ]:
def obter_colecao_mensal_sentinel(lat, lon, ano=2025, meses=[1, 2, 3, 4, 5], buffer_metros=1500, max_nuvens=20, diretorio_saida="DataSetCosta", save = True):
    if not os.path.exists(diretorio_saida):
         os.makedirs(diretorio_saida)
    aoi = ee.Geometry.Point([lon, lat]).buffer(buffer_metros).bounds()
    lista_imagens_mensais = []
    arquivos_salvos = []

    print(f"Iniciando busca de imagens para o ano {ano}...")

    for mes in meses:
        data_inicio = f"{ano}-{mes:02d}-01"
        if mes == 12:
            data_fim = f"{ano + 1}-01-01"
        else:
            data_fim = f"{ano}-{mes + 1:02d}-01"

        # Sentinel-2 Harmonized (SR - Surface Reflectance)
        colecao_mes = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(data_inicio, data_fim)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_nuvens)))

        total_imagens = colecao_mes.size().getInfo()

        if total_imagens == 0:
            lista_imagens_mensais.append(None)
            continue

        # Reduz a coleção mensal para uma única imagem usando a mediana
        #imagem_mediana = colecao_mes.sort('CLOUDY_PIXEL_PERCENTAGE').first().clip(aoi)
        imagem_mediana = colecao_mes.mean().clip(aoi)

        # 1. Extração de Bandas Brutas
        imagem_rgb = imagem_mediana.select(['B4', 'B3', 'B2'])
        imagem_nir = imagem_mediana.select(['B8'])
        imagem_mndwi = imagem_mediana.normalizedDifference(['B3', 'B11']).rename('MNDWI')
        imagem_ndvi = imagem_mediana.normalizedDifference(['B8', 'B4']).rename('NDVI')
        imagem_ndti = imagem_mediana.normalizedDifference(['B4', 'B3']).rename('NDTI')

        raw_b2 = geemap.ee_to_numpy(imagem_mediana.select('B2'), region=aoi, scale=10)
        raw_b3 = geemap.ee_to_numpy(imagem_mediana.select('B3'), region=aoi, scale=10)

        log_azul = imagem_mediana.select('B2').multiply(2000).log()
        log_verde = imagem_mediana.select('B3').multiply(2000).log()
        imagem_prof = log_azul.divide(log_verde).rename('PROF')

        img_array_rgb = convertToImageArray(imagem_rgb, aoi)
        img_array_nir = convertToImageArray(imagem_nir, aoi)
        img_array_mndwi = convertToImageArray(imagem_mndwi, aoi)
        img_array_ndvi = convertToImageArray(imagem_ndvi, aoi)
        img_array_ndti = convertToImageArray(imagem_ndti, aoi)
        img_array_prof = convertToImageArray(imagem_prof, aoi, False)

        print(f" -> Mês {mes:02d}: Mosaico gerado com sucesso ({total_imagens} imagens analisadas).")

        if save:
          nome_arquivo = f"orla_lat{lat:.4f}_lon{lon:.4f}_{ano}_{mes:02d}.npz"
          caminho_completo = os.path.join(diretorio_saida, nome_arquivo)

          np.savez_compressed(
              caminho_completo,
              lat=lat,
              lon=lon,
              mes=mes,
              ano=ano,
              rgb=img_array_rgb,
              nir=img_array_nir,
              mndwi=img_array_mndwi,
              ndvi=img_array_ndvi,
              ndti=img_array_ndti,
              prof=img_array_prof
          )

          print(f" -> Mês {mes:02d}: Mosaico salvo com sucesso em '{caminho_completo}' ({total_imagens} imagens analisadas).")
          arquivos_salvos.append(caminho_completo)

        dados_do_mes = {
            'lat': lat,
            'lon': lon,
            'ano': ano,
            'mes': mes,
            'rgb': img_array_rgb,
            'nir': img_array_nir,
            'mndwi': img_array_mndwi,
            'ndvi': img_array_ndvi,
            'ndti': img_array_ndti,
            'prof': img_array_prof
        }

        lista_imagens_mensais.append(dados_do_mes)

    registrar_coordenada_em_log(lat, lon)
    print(arquivos_salvos)
    return lista_imagens_mensais

## Obtenção de dados : Borda e escolha dos próximos alvos

In [ ]:
def _carregar_arquivo_npz(caminho_arquivo):
    try:
        with np.load(caminho_arquivo) as dados:
            return {
                'ano': int(dados['ano']),
                'mes': int(dados['mes']),
                'lat': float(dados['lat']) if 'lat' in dados else None,
                'lon': float(dados['lon']) if 'lon' in dados else None,
                'rgb': dados['rgb'].copy(),
                'nir': dados['nir'].copy(),
                'mndwi': dados['mndwi'].copy(),
                'ndvi': dados['ndvi'].copy(),
                'ndti': dados['ndti'].copy(),
                'prof': dados['prof'].copy()
            }
    except Exception as e:
        print(f"Erro ao carregar {caminho_arquivo}: {e}")
        return None

def carregar_todo_dataset(diretorio="DataSetCosta"):
    colecao_completa = []

    if not os.path.exists(diretorio):
        print(f"Diretório '{diretorio}' não encontrado.")
        return colecao_completa

    for arquivo in os.listdir(diretorio):
        if arquivo.endswith(".npz"):
            caminho_completo = os.path.join(diretorio, arquivo)
            dados = _carregar_arquivo_npz(caminho_completo)
            if dados is not None:
                colecao_completa.append(dados)

    print(f"Dataset carregado: {len(colecao_completa)} imagens totais.")
    return colecao_completa


def carregar_por_coordenada(lat, lon, diretorio="DataSetCosta"):
    colecao_filtrada = []

    if not os.path.exists(diretorio):
        return colecao_filtrada
    str_busca = f"lat{lat:.4f}_lon{lon:.4f}"

    for arquivo in os.listdir(diretorio):
        if arquivo.endswith(".npz") and str_busca in arquivo:
            caminho_completo = os.path.join(diretorio, arquivo)
            dados = _carregar_arquivo_npz(caminho_completo)
            if dados is not None:
                colecao_filtrada.append(dados)

    print(f"Foram encontrados {len(colecao_filtrada)} registros para a coordenada Lat: {lat}, Lon: {lon}.")
    return colecao_filtrada


def listar_coordenadas_processadas(diretorio="dataset_costa"):
    coordenadas_unicas = set()

    if not os.path.exists(diretorio):
        return list(coordenadas_unicas)

    for arquivo in os.listdir(diretorio):
        if arquivo.startswith("orla_") and arquivo.endswith(".npz"):
            # Exemplo: orla_lat-23.9000_lon-46.3000_2025_01.npz
            partes = arquivo.replace('.npz', '').split('_')
            try:
                lat_str = partes[1].replace('lat', '')
                lon_str = partes[2].replace('lon', '')
                # Usamos um 'set' (conjunto) para ignorar duplicações automaticamente
                coordenadas_unicas.add((float(lat_str), float(lon_str)))
            except IndexError:
                continue

    return list(coordenadas_unicas)


def registrar_coordenada_em_log(lat, lon, arquivo_registro="registro_coordenadas.txt"):
    coordenada_str = f"{lat:.4f},{lon:.4f}\n"

    if os.path.exists(arquivo_registro):
        with open(arquivo_registro, 'r') as f:
            linhas_existentes = f.readlines()
            if coordenada_str in linhas_existentes:
                return

    with open(arquivo_registro, 'a') as f:
        f.write(coordenada_str)

In [ ]:
def is_border(x, y, margem, img_shape):
    h, w = img_shape
    return x <= margem or x >= (w - 1 - margem) or y <= margem or y >= (h - 1 - margem)

def binarizacao_do_mar(img_mndwi):
  if len(img_mndwi.shape) == 3:
    img_mndwi = img_mndwi[:, :, 0]
  img_mndwi_8u = cv2.normalize(img_mndwi, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
  _, mascara_binaria = cv2.threshold(img_mndwi_8u, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
  return mascara_binaria, img_mndwi_8u

def encontrar_extremidades_da_orla(img_mndwi, img_rgb):
    if len(img_mndwi.shape) == 3:
        img_mndwi = img_mndwi[:, :, 0]

    mascara_binaria, img_mndwi_8u = binarizacao_do_mar(img_mndwi)

    contornos, _ = cv2.findContours(mascara_binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contornos:
        print("Nenhum contorno detectado.")
        return None

    maior_contorno = max(contornos, key=cv2.contourArea)

    margem = 5
    pontos_transicao = []
    pontos_praia = []

    img_shape = img_mndwi_8u.shape
    num_pontos = len(maior_contorno)

    for i in range(num_pontos):
        pt_atual = maior_contorno[i][0]
        pt_anterior = maior_contorno[i - 1][0]

        atual_na_borda = is_border(pt_atual[0], pt_atual[1], margem, img_shape)
        anterior_na_borda = is_border(pt_anterior[0], pt_anterior[1], margem, img_shape)

        if atual_na_borda != anterior_na_borda:
            ponto_extremidade = pt_atual if atual_na_borda else pt_anterior
            pontos_transicao.append(tuple(ponto_extremidade))

        if not atual_na_borda:
            pontos_praia.append(tuple(pt_atual))

    extremidade_1 = None
    extremidade_2 = None
    max_dist = 0

    if len(pontos_transicao) >= 2:
        for i in range(len(pontos_transicao)):
            for j in range(i + 1, len(pontos_transicao)):
                p1 = pontos_transicao[i]
                p2 = pontos_transicao[j]
                dist = np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)
                if dist > max_dist:
                    max_dist = dist
                    extremidade_1 = p1
                    extremidade_2 = p2

    for pt in pontos_praia:
        cv2.circle(img_rgb, pt, 2, (0, 255, 0), -1)

    extremidades_finais = []
    if extremidade_1 and extremidade_2:
        for p in [extremidade_1, extremidade_2]:
            cv2.circle(img_rgb, p, 10, (255, 0, 0), -1)
            extremidades_finais.append(p)
            print(f"Extremidade Real detectada na coordenada: X={p[0]}, Y={p[1]}")

    fig, axs = plt.subplots(1, 3, figsize=(18, 6)) # Cria 3 quadros lado a lado

    # Quadro 1: Imagem MNDWI base
    axs[0].imshow(img_mndwi_8u, cmap='gray')
    axs[0].set_title("1. MNDWI Normalizado (8-bit)")
    axs[0].axis('off')

    # Quadro 2: Imagem Limiarizada / Máscara Binária
    axs[1].imshow(mascara_binaria, cmap='gray')
    axs[1].set_title("2. Imagem Limiarizada (Máscara Otsu)")
    axs[1].axis('off')

    # Quadro 3: Imagem RGB Final
    axs[2].imshow(img_rgb)
    axs[2].set_title("3. Orla Isolada (Verde) e Extremidades")
    axs[2].axis('off')

    plt.tight_layout() # Ajusta os espaçamentos para ficar visualmente limpo
    plt.show()

    return extremidades_finais

In [ ]:
def calcular_proximos_alvos(lat_atual, lon_atual, p1, p2, passo_metros=2500):
    x1, y1 = p1
    x2, y2 = p2

    dx_pixel = x2 - x1

    dy_pixel = y1 - y2

    distancia_pixels = np.sqrt(dx_pixel**2 + dy_pixel**2)

    razao_x = dx_pixel / distancia_pixels
    razao_y = dy_pixel / distancia_pixels


    metros_por_grau_lat = 111320.0

    metros_por_grau_lon = 111320.0 * np.cos(np.radians(lat_atual))

    passo_lat_graus = passo_metros / metros_por_grau_lat
    passo_lon_graus = passo_metros / metros_por_grau_lon

    nova_lat_1 = lat_atual + (razao_y * passo_lat_graus)
    nova_lon_1 = lon_atual + (razao_x * passo_lon_graus)

    nova_lat_2 = lat_atual - (razao_y * passo_lat_graus)
    nova_lon_2 = lon_atual - (razao_x * passo_lon_graus)

    print("--- Projeção de Novas Coordenadas ---")
    print(f"Direção da Orla (Razão X, Y): ({razao_x:.2f}, {razao_y:.2f})")
    print(f"Alvo Atual: Lat {lat_atual:.6f}, Lon {lon_atual:.6f}")
    print(f"Próximo Alvo (Sentido 1): Lat {nova_lat_1:.6f}, Lon {nova_lon_1:.6f}")
    print(f"Próximo Alvo (Sentido 2): Lat {nova_lat_2:.6f}, Lon {nova_lon_2:.6f}")

    return (nova_lat_1, nova_lon_1), (nova_lat_2, nova_lon_2)

In [ ]:

steps = 3
lat_ant = 0
lon_ant = 0
lat = ini_lat
lon = ini_lon

def getImagesFromAnArea(lat, lon):
    meses_analise = list(range(1, 13))
    return obter_colecao_mensal_sentinel(lat, lon, ano=2025, meses=meses_analise, buffer_metros=1200, save = True)

def escolhaDeDirecao(conj1, conj2):
  dist_conj1 = (conj1[0] - lat_ant)**2 + (conj1[1] - lon_ant)**2
  dist_conj2 = (conj2[0] - lat_ant)**2 + (conj2[1] - lon_ant)**2
  print(f"conj1 = {conj1}")
  print(f"conj2 = {conj2}")
  print(f"lat ant = {lat_ant}")
  print(f"lon ant = {lon_ant}")
  print(f"dist conj 1 = {dist_conj1}")
  print(f"dist conj 2 = {dist_conj2}")
  if lat_ant == 0 and lon_ant == 0:
    return False
  return (dist_conj1 > dist_conj2)

for i in range(steps):
  print("# Passo")
  colecao_mar = getImagesFromAnArea(lat, lon)
  item_valido = next(item for item in colecao_mar if item is not None)
  extremidades = encontrar_extremidades_da_orla(item_valido['mndwi'], item_valido['rgb'])
  conj1, conj2 = calcular_proximos_alvos(lat, lon, extremidades[0], extremidades[1])
  conj = conj1 if escolhaDeDirecao(conj1, conj2) else conj2
  lat_ant = lat
  lon_ant = lon
  lat = conj[0]
  lon = conj[1]
  print("###########################################################")
  print("###########################################################")
  print("###########################################################")


# Exposição e recuperação dos dados

In [ ]:
def getAndShowImagesInfo(colecao_mar):

    for dados in colecao_mar:
        if dados is None:
            continue

        print(f"\n--- Processando e Exibindo: {dados['mes']} / {dados['ano']} ---")
        print(f"\n lat: {dados['lat']} lon: {dados['lon']}")
        fig, axes = plt.subplots(1, 6, figsize=(22, 5))

        # RGB
        if dados['rgb'] is not None:
            axes[0].imshow(dados['rgb'])
            axes[0].set_title("RGB (Visível)")
        else:
            axes[0].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[0].axis('off')

        # 2. Plotar NIR
        if dados['nir'] is not None:
            img_nir = dados['nir']
            if len(img_nir.shape) == 3:
                img_nir = img_nir[:, :, 0]
            axes[1].imshow(img_nir, cmap='gray')
            axes[1].set_title("NIR (B8)")
        else:
            axes[1].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[1].axis('off')

        # MNDWI
        if dados['mndwi'] is not None:
            img_mndwi = dados['mndwi']
            if len(img_mndwi.shape) == 3:
                img_mndwi = img_mndwi[:, :, 0]
            axes[2].imshow(img_mndwi, cmap='coolwarm')
            axes[2].set_title("MNDWI (Água)")
        else:
            axes[2].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[2].axis('off')

        # NDVI
        if dados['ndvi'] is not None:
            img_ndvi = dados['ndvi']
            if len(img_ndvi.shape) == 3:
                img_ndvi = img_ndvi[:, :, 0]
            axes[3].imshow(img_ndvi, cmap='YlGn')
            axes[3].set_title("NDVI (Vegetação)")
        else:
            axes[3].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[3].axis('off')

        # NDTI
        if dados['ndti'] is not None:
            img_ndti = dados['ndti']
            if len(img_ndti.shape) == 3:
                img_ndti = img_ndti[:, :, 0]
            axes[4].imshow(img_ndti, cmap='RdYlBu')
            axes[4].set_title("NDTI (Sedimentos)")
        else:
            axes[4].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[4].axis('off')

        #Barimetria
        if dados['prof'] is not None:
            img_profundidade = dados['prof']

            if len(img_profundidade.shape) == 3:
                img_profundidade = img_profundidade[:, :, 0]

            p2  = np.nanpercentile(img_profundidade, 2)
            p98 = np.nanpercentile(img_profundidade, 98)

            plot_prof = axes[5].imshow(img_profundidade, cmap='Blues', vmin=p2, vmax=p98)
            axes[5].set_title("Batimetria Relativa")

            print(f"P2: {p2:.4f} | P98: {p98:.4f}")
        else:
            axes[5].text(0.5, 0.5, 'Sem dados', ha='center', va='center')
        axes[5].axis('off')

        # Título geral da janela com o nome do mês atual
        fig.suptitle(f"Análise Multiespectral - {dados['mes']} / {dados['ano']}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()


In [ ]:
def carregar_colecao_anual(lat, lon, ano=2025, diretorio="dataset_costa"):
    colecao_mar = []

    print(f"Carregando dados salvos do diretório '{diretorio}' para o ano {ano}...")

    # Itera sobre os 12 meses do ano
    for mes in range(1, 13):
        # Reconstrói o nome exato do arquivo gerado pela sua rotina de salvamento
        nome_arquivo = f"orla_lat{lat:.4f}_lon{lon:.4f}_{ano}_{mes:02d}.npz"
        caminho_completo = os.path.join(diretorio, nome_arquivo)

        # Se o arquivo não existir (ex: mês falhou por excesso de nuvens), adicionamos None
        if not os.path.exists(caminho_completo):
            print(f" -> Mês {mes:02d}: Sem dados locais (Arquivo '{nome_arquivo}' não encontrado).")
            colecao_mar.append(None)
            continue

        # Usamos o bloco 'with' para garantir que o arquivo binário seja fechado com segurança
        with np.load(caminho_completo) as dados:
            # Reconstruindo o dicionário
            # Usamos .copy() para garantir que a matriz fique na memória RAM após o arquivo ser fechado
            dados_mes = {
                'mes': int(dados['mes']),
                'rgb': dados['rgb'].copy(),
                'nir': dados['nir'].copy(),
                'mndwi': dados['mndwi'].copy(),
                'ndvi': dados['ndvi'].copy(),
                'ndti': dados['ndti'].copy()
            }
            colecao_mar.append(dados_mes)
            print(f" -> Mês {mes:02d}: Arquivo carregado com sucesso.")

    return colecao_mar

In [ ]:

dataset = carregar_todo_dataset("/content/DataSetCosta")
getAndShowImagesInfo(dataset)

In [ ]:
!zip -r dataSetCosta.zip /content/DataSetCosta

	zip warning: missing end signature--probably not a zip file (did you
	zip warning: remember to use binary mode when you transferred it?)
	zip warning: (if you are trying to read a damaged archive try -F)

zip error: Zip file structure invalid (dataSetCosta.zip)


In [ ]:
from google.colab import files
files.download('/content/dataSetCosta.zip')
files.download('/content/registro_coordenadas.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!unzip /content/dataSetCosta.zip